In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
!pip install catboost
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from catboost import CatBoostClassifier

In [ ]:
# Task 1: Write your code here:

#read Data
Data = pd.read_csv("/kaggle/input/q3-ka-ai-2026/Q3_data.csv")
Data

In [ ]:
# Task 2: Write your code here:

#Print head
Data.head()

In [ ]:
# Task 3: Write your code here:

# Check data types and structure
Data.info()

In [ ]:
# Task 4: Write your code here:

# Descriptive statistic
Data.describe()

In [ ]:
# Task 1: Write your code here:

#check missing value
print(Data.isnull().sum())

In [ ]:
# Handle missing values
# we have many as i can drop it
Data = Data.dropna(subset=['P_2', 'B_2', 'D_142', 'D_143', 'D_144', 'D_145'])

In [ ]:
# Task 2: Write your code here:
duplicates = Data.duplicated().sum()
duplicates

In [ ]:
# Task 3: Write your code here:
df = pd.get_dummies(Data, columns=cat_cols, drop_first=True)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
numerical_cols = Data.select_dtypes(include=["int64", "float64"]).columns.drop("Target")
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

In [ ]:
# Task 5: Write your code here:
df['Target'].value_counts(normalize=True)

In [ ]:
# Fix scaled target
target_col = "Target"
df[target_col] = (df[target_col] > 0).astype(int)

In [ ]:
# Task 1: Write your code here:
X = df.drop(columns=[target_col])
y = df[target_col]

In [ ]:
# Task 2,3,4,5: Write your code here:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for tr_idx, va_idx in cv.split(X, y):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        loss_function="Logloss",
        verbose=0,
        random_seed=42
    )

    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_va)
    scores.append(f1_score(y_va, y_pred))

print("Average F1 Score across folds:", np.mean(scores))

In [ ]:
# Task 1: Write your code here:
# plot important feature
importances = model.get_feature_importance()

top_k = 20
top_idx = np.argsort(importances)[-top_k:]
top_features = X.columns[top_idx]
top_vals = importances[top_idx]

plt.figure(figsize=(10, 6))
plt.barh(top_features, top_vals)
plt.title("Top Feature Importances (CatBoost)")
plt.xlabel("Importance")
plt.show()

In [ ]:
# Task 2: Write your code here:
#print name of most important
golden_feature = X.columns[int(np.argmax(importances))]
print("Golden feature:", golden_feature)

In [ ]:
X_golden = X[[golden_feature]]

In [ ]:
for tr_idx, va_idx in cv.split(X_golden, y):
    X_tr, X_va = X_golden.iloc[tr_idx], X_golden.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

full_scores = []

for tr_idx, va_idx in cv.split(X, y):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        loss_function="Logloss",
        verbose=0,
        random_seed=42
    )
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_va)
    full_scores.append(accuracy_score(y_va, y_pred))

full_acc = np.mean(full_scores)

print("Golden-only accuracy (avg across folds):", np.mean(golden_scores))

In [ ]:
# comparison
print("Full model accuracy (avg across folds):", full_acc)
print("Accuracy difference (Full - Golden):", full_acc - np.mean(golden_scores))